# 05 · Reload and compare

**Saved base + saved adapter → fresh predictions**

First test the tiny adapter you trained in notebook 04. Then compare the existing Lightning endpoints. These are two separate experiments.

**Who does what:** Nano coordinates the checks. Python reloads your CPU adapter. Lightning base and specialist answer the live comparison question.

###  adapter reload versus serving

[PEFT loading and saving](https://huggingface.co/docs/peft/main/en/quicktour) and the separate [vLLM LoRA serving guide](https://docs.vllm.ai/en/latest/features/lora/).

finish 04, reload your CPU adapter, and compare its outputs with that adapter disabled. The later comparison requires the separately provisioned Lightning API and `train/adapters/jisr-lightning/`; see [service setup](../README.md#provision-services-before-running-the-course).

**Two experiments:** your saved CPU adapter is reattached locally using `PeftModel.from_pretrained`. The live `specialist` alias uses an existing Lightning adapter. This notebook does not deploy your CPU adapter, start a model server, or upload weights anywhere.

Source: [05 lesson source](agent/lessons/05-attach-and-serve.json). A nonzero logit difference proves that attachment changes predictions; it does not establish correctness. Keep the generated answer, expected answer, and endpoint comparison separate.


### Start here
Run the short cells in order. Helpers execute the existing lesson code and keep detailed logs expandable. All displayed results come from your run. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import LessonWalkthrough, table, bars, loss_chart, show_skill_library
lesson = LessonWalkthrough('05-attach-and-serve')

### API pattern · reload the matching base and adapter

Adapted from the [PEFT quicktour](https://huggingface.co/docs/peft/main/en/quicktour) using local workshop artifacts. Reference only; `lesson.step("reload")` executes this pattern below.

```python
from transformers import AutoModelForCausalLM
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(
    lesson.work / 'cpu/base', local_files_only=True,
)
adapted = PeftModel.from_pretrained(
    base, lesson.work / 'cpu/student-adapter', is_trainable=False,
)
```

Serving is a separate process with its own model IDs, credentials, and deployment lifecycle.


### 1 · Reload your CPU adapter

The helper loads fresh copies from disk, compares logits with the adapter enabled and disabled, and generates an answer. A nonzero change proves an effect—not correctness.

In [ ]:
lesson.step("reload")
print("Mean absolute logit change:", lesson.ns["difference"])
ids = lesson.ns["ids"]
generated = lesson.ns["generated"]
answer = bytes(x-2 for x in generated[0,len(ids[0]):].tolist() if 2 <= x < 258).decode(errors="replace")
table(["Actual CPU answer", "Expected answer"], [(answer, lesson.ns["row"]["messages"][-1]["content"])])

### 2 · Keep the two adapters straight

| Adapter | Used with |
|---|---|
| Your practice adapter | Tiny CPU model from notebook 03 |
| Existing workshop specialist | Lightning on the HTTP endpoint |

We inspect the deployed adapter’s files. We do not replace or deploy anything.

In [ ]:
lesson.step("deployed")
folder = ROOT / "train/adapters/jisr-lightning"
table(["File", "Bytes"], [(p.name, p.stat().st_size) for p in sorted(folder.iterdir()) if p.is_file()])

### 3 · Ask both Lightning aliases

The same observation goes to `base` and `specialist`. The rules-only prompt is identical for both. Compare the answers with the expected JSON; one example is not a benchmark.

In [ ]:
lesson.step("compare")
print("Observation:", lesson.ns["sample"]["messages"][1]["content"])
table(["Model", "Actual answer", "Seconds"], [(r["model"], r["answer"], r["seconds"]) for r in lesson.endpoint_calls])
display(JSON(json.loads(lesson.ns["sample"]["messages"][-1]["content"]), expanded=True))

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
manual_result = lesson.finish()
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-reload-adapter**. Open the actual instructions below.

**`inspect-adapter` → `reload-adapter` → `check-reload`**

In [ ]:
show_skill_library('05-attach-and-serve')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-reload-adapter/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-reload-adapter/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

Complete these earlier **OpenClaw demos** first: `02-training-data`, `03-build-lora`, `04-train-adapter`. This run copies their latest verified agent artifacts; its inspect action reports the inputs.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('05-attach-and-serve')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-reload-adapter/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-adapter', 'reload-adapter', 'check-reload']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the checks. Python reloads your CPU adapter. Lightning base and specialist answer the live comparison question.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** Adapter reload changes logits; base and specialist both answer.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** An adapter can change predictions without making them correct. Next, measure multiple answers.

Exit the TUI with `/exit` when done.